# Batches on WEXAC

**Setup (once per clone):** `uv sync` on a compute node (`inode`), not the login node: it compiles the Rust engine.
Batch data goes to `$MORAN_GRAPHS_DATA` (default `/home/labs/pilpel/matanyaw/moran-graphs-data`), never into the repo.

**1. Write a plan** in the cell below and run it. It prints the graphs, run counts and a rough cost, and saves `plans/<batch>.json`.
A plan holds recipes (factory + kwargs + count), r values, repeats, seeds and LSF settings, never graphs.

**2. Submit** from a terminal in the repo (light, fine on the login node):
```
uv run python -m moran_graphs.hpc submit plans/<batch>.json --dry-run   # check the plan, creates nothing
uv run python -m moran_graphs.hpc submit plans/<batch>.json             # prepare job -> worker array -> finalize job
```

**3. Watch it:**
```
uv run python -m moran_graphs.hpc status <batch>     # chunks done / running / not started, LSF job states, rough ETA
bjobs -w                                             # raw LSF view
```

**4. If something failed** (finalize reports FAIL, or workers died):
```
uv run python -m moran_graphs.hpc resume <batch>     # frees claims of dead jobs, reruns only the missing chunks, then finalizes
uv run python -m moran_graphs.hpc finalize <batch>   # rebuild the report by hand (also checks the batch again)
```

**5. Read the report** in the cells at the end. Nothing in this notebook simulates; `load_report` raises if finalize has not run.
`<batch>/report/verification.txt` and `speed.txt` are also plain text you can `cat`.

**Testing without LSF** (on an inode): `submit plans/<batch>.json --local` runs the whole chain with local processes.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from moran_graphs.plan import describe, make_plan, save_plan, spec
from moran_graphs.hpc import DATA_ROOT, load_report
from moran_graphs.engine import WILD_TYPE

## Write a plan

In [ ]:
BATCH = "2026-10-05-lsf-smoke"

specs = [
    spec("complete_graph", n_nodes=31),
    spec("cycle_graph", n_nodes=31),
    spec("star_graph", n_nodes=31),
    spec("mammalian_lung_graph", depth=4, skip_null_steps=True),
    spec("random_graph", count=5, n_nodes=31, n_edges=40),
]
plan = make_plan(BATCH, specs, r_values=[1.1, 2.0], n_repeats=100_000,
                 zoo_seed=1, batch_seed=1, n_workers=10, queue="gsla-cpu")
describe(plan)
path = save_plan(plan, f"plans/{BATCH}.json")
print(f"submit with: uv run python -m moran_graphs.hpc submit {path}")

## Read a finished batch

In [ ]:
rep = load_report(BATCH)
print(rep["verification"]["overall"])
for c in rep["verification"]["checks"]:
    print(f"[{c['status']:<4}] {c['check']:<15} {c['message']}")
results = rep["results"]
results[["name", "r", "n", "rho", "rho_lo", "rho_hi", "steps_mean_given_fixed", "n_censored"]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for r, part in results.groupby("r"):
    part = part.sort_values("name")
    ax.errorbar(part["name"], part["rho"], yerr=[part["rho"] - part["rho_lo"], part["rho_hi"] - part["rho"]],
                fmt="o", capsize=3, label=f"r = {r}")
ax.set(ylabel="fixation probability (95% Wilson CI)")
ax.tick_params(axis="x", rotation=45)
ax.legend(frameon=False)
plt.tight_layout()

## How the batch ran

In [ ]:
for key, value in rep["speed"].items():
    print(f"{key:28s} {value}")
t = rep["chunk_timing"]
fig, (ax_w, ax_d) = plt.subplots(1, 2, figsize=(13, 4))
t0 = t["start"].min()
for k, (worker, part) in enumerate(t.groupby(["job", "index"])):
    ax_w.barh(k, part["end"] - part["start"], left=part["start"] - t0, height=0.8, color="#2a78d6")
ax_w.set(xlabel="seconds since the first chunk started", ylabel="worker", title="Chunks per worker (gaps are idle time)")
ax_d.hist(t["end"] - t["start"], bins=30, color="#2a78d6")
ax_d.set(xlabel="chunk wall time (s)", ylabel="chunks", title="Chunk durations (tune chunk_target_steps)")
plt.tight_layout()